# Paper Metadata

One tidy row per paper. Primary key: `paper_id` (OpenAlex `W…`, shared with every other paper output in
`/project/jevans/Dawoon/Science of Science/OpenAlex/output/`).

## Inputs
```
/project/jevans/renli_shared/OpenAlex_2026_Jan_16_Renly_parquet/        # read-only snapshot
  works/works/part_*.parquet           # id, publication_year, type, cited_by_count, is_retracted -> the row set
  works/works_semantic/part_*.parquet  # primary_topic_json                                       -> primary_topic_*
  works/topics/part_*.parquet          # work_id, topic_id, score (<= 3 scored topics per work)    -> field / subfield / topic
  topics.csv.gz                        # topic -> subfield -> field hierarchy (4,516 topics)
OpenAlex/output/referenced_works_w_year/part_*.parquet   # one row per reference edge                -> ref_count
OpenAlex/cache/paper_journal.parquet   # work -> journal, is_journal    (oa.build_journal)
OpenAlex/cache/paper_fos.parquet       # work -> FoS_0, FoS_rep, domain (oa.build_fos)
```
The four partitioned tables hold the same works in the same `part_XXXX` (1,334 each), so every join is local
to one partition. `notebook/referenced_works_w_year.ipynb` must have run first (edge table + work map).

## Output columns
| column | type | meaning |
|---|---|---|
| `paper_id` | string | `W…` |
| `year` | float64 | `publication_year` (float because ~6% of works have none; kept as in earlier builds) |
| `doctype` | string | OpenAlex `type` |
| `ref_count` | int32 | outgoing reference edges in the edge table |
| `journal`, `is_journal` | string, bool | primary-location source name and whether it is a journal; null when the work has no source |
| `author_list` | null | always null, kept for schema compatibility — authors are in `paper_author.parquet` |
| `FoS_0` | string | `;`-joined distinct fields of all the work's topics, best topic first |
| `FoS_rep`, `domain` | string | field and domain of the top-scored topic |
| `primary_topic_field` | string | field of OpenAlex's own primary topic (`primary_topic_json`) |
| `primary_topic_subfield` | string | subfield of the primary topic |
| `primary_topic_topic` | string | the primary topic itself (name from `topics.csv.gz`, looked up by id) |
| `field` | list&lt;string&gt; | **all** fields of the work's topics, distinct, best topic score first |
| `subfield` | list&lt;string&gt; | **all** subfields of the work's topics, distinct by name, best topic score first |
| `topic` | list&lt;string&gt; | **all** topics of the work (one entry per scored topic), best score first |
| `cited_by_count` | int64 | OpenAlex `cited_by_count` |
| `is_retracted` | bool | OpenAlex `is_retracted` |

`field` / `subfield` / `topic` are null (not empty) when the work has no topic. Eight subfield names are
shared by two subfields (e.g. *Genetics* under Biochemistry vs. Medicine); `subfield` dedupes by name, so such
a pair appears once. Ids and scores are in `paper_topics.parquet`.

Output → `/project/jevans/Dawoon/Science of Science/OpenAlex/output/paper_metadata.parquet`, one row group per
snapshot partition, rows in snapshot order (the same order as earlier builds).

> Streams partition by partition through a `ParquetWriter`: memory is the two cache indexes (~20 GB) plus one
> partition. The file is written to `paper_metadata.parquet.tmp`, every pre-existing column is compared row for
> row with the previous file, and only then is it swapped in. Smoke test: `NB_PART_LIMIT=n` writes the first
> n partitions to `paper_metadata.smoke.parquet` and leaves the real file alone.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/OpenAlex')
import oa_common as oa
ROOT = oa.BASE; OUT = oa.OUT
print('snapshot:', oa.ROOT)
OUT_FP = f'{OUT}/paper_metadata.parquet'
# Smoke test: NB_PART_LIMIT=n -> first n partitions only, written next to the real file, which is not touched.
PART_LIMIT = int(os.environ.get('NB_PART_LIMIT', '0') or 0)
NEW_FP = f'{OUT}/paper_metadata.smoke.parquet' if PART_LIMIT else f'{OUT_FP}.tmp'
# Every column that existed before must match the previous file row for row before it is replaced.
# Set False only when a column's definition is changed on purpose.
REQUIRE_PARITY = True
# author_list is no longer built here: the old RUN_AUTHORS scan over works_au_affs_fixed.csv.gz overcounted
# authors (see paper_author.ipynb), and paper_author.parquet is the per-paper author table.
assert oa.have_consolidated(), (
    'run notebook/referenced_works_w_year.ipynb first — it builds the map and edge table')
oa.summary()
WP, SP, TP, REF = oa.parts('works'), oa.parts('works_semantic'), oa.parts('topics'), oa.ref_parts()
assert len(WP) == len(SP) == len(TP) == len(REF), 'works, works_semantic, topics and edge partitions must line up'
if PART_LIMIT:
    WP, SP, TP, REF = WP[:PART_LIMIT], SP[:PART_LIMIT], TP[:PART_LIMIT], REF[:PART_LIMIT]
    print(f'SMOKE: first {PART_LIMIT} partitions -> {NEW_FP}')
print(f'{len(WP)} partitions -> {NEW_FP}')

## 1. Lookups: journal + FoS caches (sorted work codes), topic hierarchy

In [ ]:
%%time
# 1. The journal and FoS caches are keyed by work_id but not partitioned like the snapshot. Each is held once
#    as sorted int64 work codes + dictionary-coded columns and looked up per partition with searchsorted;
#    a merge against ~290M rows per partition would rebuild a hash table 1,334 times.
oa.build_journal()
oa.build_fos()

def wcode(ids, strict=True):
    """Arrow 'W123' (or '.../W123') -> int64 123. strict: every id must be a W-number (the caches);
    otherwise a malformed id (the snapshot has one placeholder, 'W-1') becomes -1, which matches no cache
    entry -- the null journal / FoS a string join would have given it."""
    ids = pc.replace_substring_regex(ids, pattern=r'^.*/', replacement='')
    ok = pc.fill_null(pc.match_substring_regex(ids, r'^W\d+$'), False)
    if strict:
        assert pc.all(ok).as_py(), 'non-W work id'
        return pc.cast(pc.utf8_slice_codeunits(ids, 1), pa.int64()).to_numpy()
    return pc.cast(pc.if_else(ok, pc.utf8_slice_codeunits(ids, 1), '-1'), pa.int64()).to_numpy()

class CacheIndex:
    """A work_id-keyed cache parquet as sorted work codes + per-column category codes."""
    def __init__(self, fp, str_cols, bool_cols=()):
        t = pq.read_table(fp, columns=['work_id', *str_cols, *bool_cols], read_dictionary=list(str_cols))
        code = wcode(t['work_id'])
        order = np.argsort(code, kind='stable')
        self.code = code[order]
        assert (np.diff(self.code) > 0).all(), f'{os.path.basename(fp)}: duplicate work_id'
        self.cols = {}
        for c in str_cols:
            s = t[c].to_pandas()                  # Categorical: per-row-group dictionaries unified
            self.cols[c] = (s.cat.codes.to_numpy()[order],
                            pa.array(s.cat.categories.to_numpy(dtype=object), pa.string()))
        for c in bool_cols:
            assert t[c].null_count == 0, f'{c} has nulls in the cache'
            self.cols[c] = (t[c].to_numpy()[order], None)
        del t

    def take(self, code_p):
        """{column: Arrow array} for the works `code_p`, in that order; null where a work is not cached."""
        i = np.minimum(np.searchsorted(self.code, code_p), len(self.code) - 1)
        hit = self.code[i] == code_p
        out = {}
        for c, (v, cats) in self.cols.items():
            if cats is None:
                out[c] = pa.array(v[i], mask=~hit)
            else:
                k = np.where(hit, v[i], -1)
                out[c] = pc.cast(pa.DictionaryArray.from_arrays(pa.array(k, mask=k < 0), cats), pa.string())
        return out

JR = CacheIndex(oa.JOURNAL_PQ, ['journal'], ['is_journal'])
FS = CacheIndex(oa.FOS_PQ, ['FoS_0', 'FoS_rep', 'domain'])
print(f'journal index {len(JR.code):,} works | FoS index {len(FS.code):,} works')

# topic -> subfield -> field names. Names come from topics.csv.gz (by topic id), not from primary_topic_json,
# whose json.dumps escapes the 12 non-ASCII topic names as \uXXXX.
tp = pd.read_csv(f'{oa.ROOT}/topics.csv.gz', dtype=str,
                 usecols=['id', 'display_name', 'subfield_id', 'subfield_display_name',
                          'field_id', 'field_display_name'])
TOPIC_IDX = pd.Index(oa.norm_id(tp['id']))
assert TOPIC_IDX.is_unique and tp.notna().all().all()
T_NAME  = tp['display_name'].to_numpy(dtype=object)
T_SUB   = tp['subfield_display_name'].to_numpy(dtype=object)
T_FIELD = tp['field_display_name'].to_numpy(dtype=object)
print(f"{len(tp):,} topics -> {tp.subfield_id.nunique()} subfields ({tp.subfield_display_name.nunique()} names) "
      f"-> {tp.field_id.nunique()} fields")

## 2. Assemble partition by partition → ParquetWriter

In [ ]:
%%time
# 2. One snapshot partition -> one row group, in snapshot order.
SCHEMA = pa.schema([
    ('paper_id', pa.string()), ('year', pa.float64()), ('doctype', pa.string()), ('ref_count', pa.int32()),
    ('journal', pa.string()), ('is_journal', pa.bool_()), ('author_list', pa.null()),
    ('FoS_0', pa.string()), ('FoS_rep', pa.string()), ('domain', pa.string()),
    ('primary_topic_field', pa.string()), ('primary_topic_subfield', pa.string()),
    ('primary_topic_topic', pa.string()),
    ('field', pa.list_(pa.string())), ('subfield', pa.list_(pa.string())), ('topic', pa.list_(pa.string())),
    ('cited_by_count', pa.int64()), ('is_retracted', pa.bool_()),
])
# primary_topic_json is json.dumps output: {"id": ".../T…", "display_name": …, "subfield": {…}, "field": {…}, …}.
# RE2 regexes instead of json.loads on 372M rows. The subfield block always precedes the field block.
PT_TOPIC_RE = r'^\{\s*"id":\s*"https://openalex\.org/(?P<topic_id>T\d+)"'
PT_SUBFIELD_RE = (r'"subfield":\s*\{\s*"id":\s*"https://openalex\.org/subfields/(?P<subfield_id>\d+)",'
                  r'\s*"display_name":\s*"(?P<subfield>[^"]*)"')
PT_FIELD_RE = (r'"field":\s*\{\s*"id":\s*"https://openalex\.org/fields/(?P<field_id>\d+)",'
               r'\s*"display_name":\s*"(?P<field>[^"]*)"')

def list_col(pos, values, n):
    """`values` grouped by row position `pos` (ascending) -> list<string> of length n, null for rows with none."""
    counts = np.bincount(pos, minlength=n)
    offsets = np.zeros(n + 1, dtype=np.int32)
    np.cumsum(counts, out=offsets[1:])
    return pa.ListArray.from_arrays(pa.array(offsets), pa.array(values, pa.string()),
                                    mask=pa.array(counts == 0))

def n_agree(a, b):
    """(rows where both are present, rows where they are equal) for two pandas Series."""
    both = a.notna().to_numpy() & b.notna().to_numpy()
    return int(both.sum()), int((a.to_numpy()[both] == b.to_numpy()[both]).sum())

stats = dict(works=0, journal=0, fos=0, primary_topic_field=0, primary_topic_subfield=0, primary_topic_topic=0,
             with_topics=0, orphan_topic_rows=0, unknown_topic_rows=0, unknown_primary=0,
             rep_pt_both=0, rep_pt_agree=0, top_pt_both=0, top_pt_agree=0, fos0_field_both=0, fos0_field_agree=0,
             malformed_ids=0)
n_topics = pd.Series(dtype='int64')
subfields_seen = set()

if os.path.exists(NEW_FP):
    os.remove(NEW_FP)
writer = pq.ParquetWriter(NEW_FP, SCHEMA, compression='snappy')
t0 = time.time()
for i, (fw, fr, fs_, ft) in enumerate(zip(WP, REF, SP, TP)):
    # -- row set
    w = pq.read_table(fw, columns=['id', 'publication_year', 'type', 'cited_by_count', 'is_retracted']).to_pandas()
    w = w.rename(columns={'id': 'paper_id', 'publication_year': 'year', 'type': 'doctype'})
    w['paper_id'] = oa.norm_id(w['paper_id'])
    w['year'] = w['year'].astype('float64')
    pos_idx = pd.Index(w['paper_id'])
    assert pos_idx.is_unique, f'{os.path.basename(fw)}: duplicate work id'
    n = len(w)
    # -- reference count: one edge row per reference, so a count per work_id is the count
    rc = pq.read_table(fr, columns=['work_id']).group_by('work_id').aggregate([([], 'count_all')])
    ref = np.zeros(n, dtype=np.int32)
    j = pos_idx.get_indexer(oa.norm_id(rc['work_id'].to_pandas()))
    ref[j[j >= 0]] = rc['count_all'].to_numpy()[j >= 0]
    w['ref_count'] = ref
    # -- primary topic (works_semantic, aligned partition)
    s = pq.read_table(fs_, columns=['work_id', 'primary_topic_json'])
    js = s['primary_topic_json']
    pt = pd.DataFrame({'paper_id': oa.norm_id(s['work_id'].to_pandas()),
                       'primary_topic_field': pc.struct_field(pc.extract_regex(js, PT_FIELD_RE), 'field').to_pandas(),
                       'primary_topic_subfield': pc.struct_field(pc.extract_regex(js, PT_SUBFIELD_RE), 'subfield').to_pandas(),
                       'primary_topic_id': pc.struct_field(pc.extract_regex(js, PT_TOPIC_RE), 'topic_id').to_pandas()})
    k = TOPIC_IDX.get_indexer(pt['primary_topic_id'])
    stats['unknown_primary'] += int(((k < 0) & pt['primary_topic_id'].notna().to_numpy()).sum())
    pt['primary_topic_topic'] = np.where(k >= 0, T_NAME[k], None)
    w = w.merge(pt.drop(columns='primary_topic_id'), on='paper_id', how='left')   # left merge keeps w's order
    assert len(w) == n, f'{os.path.basename(fs_)}: duplicate work_id in works_semantic'
    # -- all topics (works/topics, aligned partition) -> field / subfield / topic lists, best score first.
    #    The two-key sort is a lexsort, so tied scores keep snapshot order.
    t = pq.read_table(ft, columns=['work_id', 'topic_id', 'score']).to_pandas()
    t['pos'] = pos_idx.get_indexer(oa.norm_id(t['work_id']))
    t['k'] = TOPIC_IDX.get_indexer(oa.norm_id(t['topic_id']))
    stats['orphan_topic_rows'] += int((t['pos'] < 0).sum())
    stats['unknown_topic_rows'] += int(((t['pos'] >= 0) & (t['k'] < 0)).sum())
    t = t[(t['pos'] >= 0) & (t['k'] >= 0)].sort_values(['pos', 'score'], ascending=[True, False])
    pos, k = t['pos'].to_numpy(), t['k'].to_numpy()
    sub = pd.DataFrame({'pos': pos, 'name': T_SUB[k]}).drop_duplicates()     # keeps the first = best-scored
    fld = pd.DataFrame({'pos': pos, 'name': T_FIELD[k]}).drop_duplicates()
    lists = {'topic': list_col(pos, T_NAME[k], n),
             'subfield': list_col(sub['pos'].to_numpy(), sub['name'].to_numpy(), n),
             'field': list_col(fld['pos'].to_numpy(), fld['name'].to_numpy(), n)}
    # -- journal + FoS from the cache indexes
    code_p = wcode(pa.array(w['paper_id'], pa.string()), strict=False)
    stats['malformed_ids'] += int((code_p < 0).sum())
    cols = {**JR.take(code_p), **FS.take(code_p), **lists, 'author_list': pa.nulls(n)}
    for c in ('paper_id', 'year', 'doctype', 'ref_count', 'primary_topic_field', 'primary_topic_subfield',
              'primary_topic_topic', 'cited_by_count', 'is_retracted'):
        cols[c] = pa.Array.from_pandas(w[c])
    writer.write_table(pa.Table.from_arrays([cols[f.name].cast(f.type) for f in SCHEMA], schema=SCHEMA))
    # -- counters
    stats['works'] += n
    stats['journal'] += n - cols['journal'].null_count
    stats['fos'] += n - cols['FoS_rep'].null_count
    for c in ('primary_topic_field', 'primary_topic_subfield', 'primary_topic_topic'):
        stats[c] += int(w[c].notna().sum())
    stats['with_topics'] += n - lists['topic'].null_count
    n_topics = n_topics.add(pd.Series(np.bincount(pos, minlength=n)).value_counts(), fill_value=0)
    subfields_seen.update(pc.unique(pc.list_flatten(lists['subfield'])).to_pylist())
    for key, a, b in (('rep_pt', cols['FoS_rep'].to_pandas(), w['primary_topic_field']),
                      ('top_pt', pc.list_element(lists['topic'], 0).to_pandas(), w['primary_topic_topic']),
                      ('fos0_field', cols['FoS_0'].to_pandas(), pc.binary_join(lists['field'], ';').to_pandas())):
        nb_, na_ = n_agree(a, b)
        stats[f'{key}_both'] += nb_; stats[f'{key}_agree'] += na_
    del w, rc, s, js, pt, t, sub, fld, lists, cols
    if (i + 1) % 100 == 0 or i + 1 == len(WP):
        el = time.time() - t0
        print(f'  {i+1}/{len(WP)} partitions | {stats["works"]:,} works | {el/60:.1f} min '
              f'(eta {el/(i+1)*(len(WP)-i-1)/60:.0f} min)', flush=True)
        gc.collect()
writer.close()
print(f'WROTE {NEW_FP}  ({stats["works"]:,} rows, {len(SCHEMA)} cols, {os.path.getsize(NEW_FP)/1e9:.2f} GB)')

## 3. Checks, parity with the previous file, swap in

In [ ]:
# 3. Counters (nothing is read back in full), then a row-by-row comparison of every pre-existing column
#    with the previous file, then the atomic swap. A failed check leaves the new file at NEW_FP.
W = stats['works']
pct = lambda key: f'{stats[key] / W * 100:.1f}%'
agree = lambda key: f"{stats[key + '_agree'] / max(stats[key + '_both'], 1) * 100:.3f}% (n={stats[key + '_both']:,})"
print(f"works {W:,} | journal {pct('journal')} | FoS {pct('fos')} | primary_topic_field {pct('primary_topic_field')}"
      f" | primary_topic_subfield {pct('primary_topic_subfield')} | primary_topic_topic {pct('primary_topic_topic')}"
      f" | topic list {pct('with_topics')}")
print('topics per work:', {int(k): f'{v / W * 100:.1f}%' for k, v in n_topics.sort_index().items()})
print(f'distinct subfield names in `subfield`: {len(subfields_seen)}')
print(f"FoS_rep == primary_topic_field     : {agree('rep_pt')}")
print(f"topic[0] == primary_topic_topic    : {agree('top_pt')}")
print(f"FoS_0 == ';'.join(field)           : {agree('fos0_field')}")
print(f"orphan topic rows (work not in its partition) {stats['orphan_topic_rows']:,} | topic ids missing from "
      f"topics.csv.gz {stats['unknown_topic_rows']:,} | primary topic ids missing {stats['unknown_primary']:,} | "
      f"malformed work ids (no cache lookup) {stats['malformed_ids']:,}")
assert stats['orphan_topic_rows'] == 0, 'partition alignment broken: a topic row landed outside its work partition'
pf = pq.ParquetFile(NEW_FP)
assert pf.metadata.num_rows == W
print(pf.schema_arrow)

OLD_COLS = ['paper_id', 'year', 'doctype', 'ref_count', 'journal', 'is_journal', 'FoS_0', 'FoS_rep', 'domain',
            'primary_topic_field', 'primary_topic_subfield', 'cited_by_count', 'is_retracted']

def parity(new_fp, old_fp, cols):
    """Rows that differ per column: each row group of the new file vs the same rows of the old file."""
    old, new = pq.ParquetFile(old_fp), pq.ParquetFile(new_fp)
    cols = [c for c in cols if c in old.schema_arrow.names]
    it = old.iter_batches(batch_size=1_000_000, columns=cols)
    buf = old.schema_arrow.empty_table().select(cols)
    bad = dict.fromkeys(cols, 0)
    for g in range(new.num_row_groups):
        a = new.read_row_group(g, columns=cols)
        while buf.num_rows < a.num_rows:
            buf = pa.concat_tables([buf, pa.Table.from_batches([next(it)])])
        b, buf = buf.slice(0, a.num_rows), buf.slice(a.num_rows)
        for c in cols:
            if not a[c].equals(b[c]):
                same = pc.or_kleene(pc.equal(a[c], b[c]), pc.and_(pc.is_null(a[c]), pc.is_null(b[c])))
                bad[c] += a.num_rows - int(pc.sum(pc.fill_null(same, False)).as_py() or 0)
    return bad

if os.path.exists(OUT_FP):
    t1 = time.time()
    n_old = pq.ParquetFile(OUT_FP).metadata.num_rows
    bad = parity(NEW_FP, OUT_FP, OLD_COLS)
    print(f'\nprevious file {n_old:,} rows | rows differing per column ({(time.time() - t1) / 60:.1f} min): {bad}')
    if REQUIRE_PARITY:
        assert not any(bad.values()), 'a pre-existing column changed -- inspect NEW_FP (or set REQUIRE_PARITY=False)'
        assert PART_LIMIT or n_old == W, f'row count changed: {n_old:,} -> {W:,}'
if not PART_LIMIT:
    os.replace(NEW_FP, OUT_FP)
    print(f'REPLACED {OUT_FP}  ({os.path.getsize(OUT_FP) / 1e9:.2f} GB)')

## 4. Example

In [ ]:
# Ten rows via iter_batches: the file is never materialised in full.
pd.set_option('display.max_colwidth', 80); pd.set_option('display.width', 250)
ex = next(pq.ParquetFile(NEW_FP if PART_LIMIT else OUT_FP).iter_batches(batch_size=10)).to_pandas()
display(ex[['paper_id', 'year', 'doctype', 'ref_count', 'journal', 'FoS_rep', 'cited_by_count']])
display(ex[['paper_id', 'primary_topic_field', 'primary_topic_subfield', 'primary_topic_topic',
            'field', 'subfield', 'topic']])